# Exploratory questions: how Madrid's bus fleet works and how it could transition
**IE Sustainability Datathon (Kearney) · Supporting notebook 03**

Works through the six questions the team agreed on the whiteboard. It uses the same setup cells and `data_raw/` files as the main notebook. Findings feed the main notebook and the report. **This notebook never writes `File_1.csv`–`File_4.csv`.**

Each section is tagged **[ONCE]** (Madrid only) or **[PER CITY]** (repeated for each benchmark city), as on the board.

## Scope agreed on the whiteboard

| # | Question | Scope | Sub-questions |
|---|---|---|---|
| Q1 | How does the current fleet work? | ONCE | **Buses:** km (range), type (standard / articulated), capacity, gas / electric, age · **Lines:** electric / gas, # buses, riders, km, schedule · **Depots:** grid capacity, electrical output |
| Q2 | Apply the Madrid setup to each benchmark city | PER CITY | KPI differences · time / phases (% electrified = # zero-emission lines ÷ total lines) · gap between Madrid and the city |
| Q3 | Why were certain lines electrified first? | ONCE | Features: depot, similarity of lines, distance, traffic, # buses, riders |
| Q4 | How would we transition the buses? | ONCE | **a)** grid / depots: depot information, gap to electrification per depot, route impact / coverage · **b)** which routes are replaced: phases, route impact (buses, bus metrics) |
| Q5 | What does implementing electrification mean? | ONCE | Finance / costs: bus age and retirement, gas, labour |
| Q6 | Scenario comparison | ONCE | Depot → line · area / zones · bus ages / lifetime · distance · emissions · ridership |

**Data groups:** Madrid's existing setup (km, fleet, lines, depots and grid capacity, emissions) · benchmark cities · external signals (traffic, incline) · financials (age / retirement, gas, labour).
**Constraints:** zero-emission routes · ridership.
**Out of scope** (crossed out on the board): weather, trends / patterns, disposal, backups.

## 0 · Setup
Same helpers and fixed parameters as the main notebook (`fetch`, `assume`, `check`, `P`, GTFS helpers). Keep them identical: if you change one here, change it in notebook 01 too.

In [ ]:
# Environment: runs as-is in Google Colab (Python 3.10+).
!pip -q install pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")

In [ ]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

In [ ]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

In [ ]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## 1 · Data
Same sources as main notebook §1. `fetch()` reuses files already in `data_raw/`.

In [ ]:
# Direct file URLs, resolved from each landing page via the portal's CKAN API (brief section 4.1).
EMT = "https://datos.emtmadrid.es/dataset"
SOURCES = {
    # filename: (url, note)
    "emt_gtfs.zip": (f"{EMT}/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip",
                     "EMT GTFS, rolling daily feed — https://datos.madrid.es/dataset/900030-0-emt-autobus-gtfs"),
    "emt_supply_2026.csv": (f"{EMT}/03c64a21-3ee8-461a-9f9a-cb4c727a41be/resource/8400b44b-ca21-45b9-b3cf-7a15b754b739/download/2026busoferta.csv",
                     "Buses in service per day, line and hour band, 2026 — https://datos.madrid.es/dataset/300596-0-emt-lineas-autobuses-tramos"),
    "emt_fleet_history.csv": (f"{EMT}/071db69e-5e59-4ceb-a6da-92d895f56b62/resource/48d59bf3-93a8-48b7-96bd-cf78cc7c902b/download/vehicles.csv",
                     "Fleet history (vehicle, dates, fuel, size) — https://datos.madrid.es/dataset/300241-0-emt-flota-autobuses"),
    "emt_calendar.csv": (f"{EMT}/405a6ed0-9464-4d04-9edb-939031717b20/resource/b8cceab8-9173-4eec-8431-21f5e9e25834/download/calendar.csv",
                     "Operating calendar 2024-2027 — https://datos.madrid.es/dataset/900022-0-calendario-emt-autobus"),
    "emt_ridership_2026.csv": (f"{EMT}/29b27217-429f-44a0-a90a-2db7f13529b6/resource/07622d4b-6c64-43ea-a398-69134d6128bc/download/2026demdialineafranjaanual.csv",
                     "Optional: daily riders per line and hour band, 2026 — https://datos.madrid.es/dataset/900028-0-emt-demanda-autobus"),
}
PATHS = {fn: fetch(url, fn, note) for fn, (url, note) in SOURCES.items()}
show_register()

In [ ]:
# Load the tables used below. Fleet history: one row per vehicle record; a vehicle can have several records.
supply = pd.read_csv(PATHS["emt_supply_2026.csv"], sep=";", dtype={"Linea": str})
riders = pd.read_csv(PATHS["emt_ridership_2026.csv"], sep=";", dtype={"Linea": str})
emt_cal = pd.read_csv(PATHS["emt_calendar.csv"], dtype=str)
fleet_hist = pd.read_csv(PATHS["emt_fleet_history.csv"], dtype=str).apply(lambda c: c.str.strip())
fleet_hist["ini"] = pd.to_datetime(fleet_hist.dateIni.str[:10])
fleet_hist["end"] = pd.to_datetime(fleet_hist.dateEnd.str[:10].replace("9999-12-31", "2262-04-11"))

ZE_FUELS = assume("Zero-emission fuel codes", ("EL", "H"), "A",
                  "Fleet-history fuelType: EL battery-electric, H hydrogen. CNG/biomethane never count (README rule)")

def active_fleet(date):
    """Vehicles in service on `date`: one row per vehicle, with its first entry into service."""
    a = fleet_hist[(fleet_hist.ini <= date) & (fleet_hist.end > date)].sort_values("ini").drop_duplicates("vehicleId", keep="last")
    first = fleet_hist.groupby("vehicleId").ini.min().rename("first_in_service")
    a = a.join(first, on="vehicleId")
    a["age_years"] = (pd.Timestamp(date) - a.first_in_service).dt.days / 365.25
    a["zero_emission"] = a.fuelType.isin(ZE_FUELS)
    return a

print(f"Fleet history: {len(fleet_hist)} records, {fleet_hist.vehicleId.nunique()} vehicles, last entry {fleet_hist.ini.max().date()}")

## Timeline anchors (whiteboard)
| Anchor | Year | Meaning |
|---|---|---|
| Must do | computed below | Year the fleet is fully zero-emission at the **2021–25 purchase pace** (unadjusted S0) |
| Mid | 2035 | End of the File 4 horizon |
| Ideal | 2030 | Brief scenario S1: 100 % zero-emission by 31 Dec 2030 |
| Wildcard | open | Candidate: Madrid's official 2033 target (City of Madrid, Aug 2023) |

The pace here is **unadjusted** and the fleet history's last entry is mid-2025, so main notebook §3 reconciles the 31 Dec 2025 baseline against EMT's accounts; use its figure in the report.

In [ ]:
snap = {d: active_fleet(d) for d in ("2020-12-31", "2025-12-31")}
pace_tbl = pd.DataFrame({d: dict(fleet_total=len(a), zero_emission=int(a.zero_emission.sum()), not_zero_emission=int((~a.zero_emission).sum()))
                         for d, a in snap.items()}).T
display(pace_tbl)

ze_per_year = assume("Zero-emission buses added per year, 2021-25 (unadjusted)",
                     (pace_tbl.loc["2025-12-31", "zero_emission"] - pace_tbl.loc["2020-12-31", "zero_emission"]) / 5,
                     "D", "Fleet history, active vehicles on 31 Dec 2020 and 31 Dec 2025", "Timeline", "Main notebook §3 reconciliation")
remaining = pace_tbl.loc["2025-12-31", "not_zero_emission"]
must_do_year = 2025 + int(np.ceil(remaining / ze_per_year))

TIMELINE = pd.DataFrame([
    ("Ideal", assume("Ideal year (S1)", 2030, "V", "Brief scenario S1")),
    ("Wildcard", assume("Wildcard year", 2033, "V", "City of Madrid zero-emission bus target, Aug 2023")),
    ("Mid", assume("Mid year", 2035, "V", "Brief File 4 horizon")),
    ("Must do", assume("Must-do year (S0 pace)", must_do_year, "D", f"{remaining} non-zero-emission buses / {ze_per_year:.0f} per year")),
], columns=["anchor", "year"])
display(TIMELINE)

check("Fleet split adds up", (pace_tbl.zero_emission + pace_tbl.not_zero_emission == pace_tbl.fleet_total).all())
check("Zero-emission fleet grew 2020-25", ze_per_year > 0, f"{ze_per_year:.1f} buses/year")

## Q1 · How does the current fleet work? [ONCE]
### 1a · Buses
Fleet in service on 31 Dec 2025 by fuel and size code, with age. `typeCapacity` codes are not decoded yet (01 is by far the most common, likely standard 12 m): **TODO(owner: baseline)** confirm the code list and map to Standard / Articulated.

In [ ]:
fleet25 = snap["2025-12-31"]
display(pd.crosstab(fleet25.fuelType, fleet25.typeCapacity, margins=True, margins_name="total"))
display(fleet25.groupby("fuelType").agg(buses=("vehicleId", "count"),
                                        median_capacity=("capacity", lambda c: pd.to_numeric(c, errors="coerce").median()),
                                        mean_age=("age_years", "mean"), oldest=("age_years", "max")).round(1))

In [ ]:
import plotly.express as px
fig = px.histogram(fleet25, x="age_years", color="fuelType", nbins=20, barmode="stack",
                   title="Fleet in service on 31 Dec 2025: age by fuel type", labels={"age_years": "age (years)"})
fig.show()

### 1b · Lines
Per line on regular working days (`LA`) in 2026: average peak buses in service (max over hour bands, averaged over days) and average daily riders. **Still TODO:** electric / gas status per line (no bus-to-line link in the open data; evidence says 45 electric lines + line 145 hydrogen) and km / schedule from GTFS, which needs `frequencies.txt` expanded (main notebook §4).

In [ ]:
wd = supply[supply.TipoDiaMo == "LA"]
peak = wd.groupby(["Linea", "FechaSer"]).NumCCReal.max().groupby("Linea").mean().rename("peak_buses_avg")

working_days = set(emt_cal.loc[emt_cal.TipoDiaEs == "LA", "Fecha"])
r = riders[riders.Fecha.isin(working_days)]
riders_day = r.groupby(["Linea", "Fecha"]).TotalViajeros.sum().groupby("Linea").mean().rename("riders_per_day_avg")

lines_tbl = pd.concat([peak, riders_day], axis=1)
lines_tbl["riders_per_peak_bus"] = lines_tbl.riders_per_day_avg / lines_tbl.peak_buses_avg
display(lines_tbl.sort_values("riders_per_day_avg", ascending=False).round(1).head(15))
print(f"{len(lines_tbl)} lines | in supply only: {lines_tbl.riders_per_day_avg.isna().sum()} | in ridership only: {lines_tbl.peak_buses_avg.isna().sum()}")
check("Most lines appear in both supply and ridership", lines_tbl.notna().all(axis=1).mean() > 0.9,
      f"{lines_tbl.notna().all(axis=1).mean():.0%} matched")

### 1c · Depots
Grid capacity and electrical output per depot. Only Sanchinarro is evidenced (8 MW main + 5 MW backup); the other four need a documented proxy. Shared with main notebook §5–§6.

In [ ]:
# TODO(owner: depots) depot table (name, coordinates, connection MW, evidence); reuse main notebook §5-§6 once written

## Q2 · Apply the Madrid setup to each benchmark city [PER CITY]
For Helsinki, Berlin, Paris and Brussels: KPI differences, time / phases, and the gap to Madrid. Needs notebook 02's `benchmark_gtfs_intermediate.csv` plus cited fleet and target figures.

**KPI definition from the board:** % electrified = # zero-emission lines ÷ total # lines (alongside the fleet share used in File 1).

In [ ]:
# TODO(owner: benchmark) per-city KPI table: n_lines, ze_lines, pct_lines_electrified, ze_share_pct, target year, years to target at current pace; gap vs Madrid

## Q3 · Why were certain lines electrified first? [ONCE]
Compare already-electrified lines with the rest on: depot, similarity of lines, distance, traffic, # buses, riders. External signals: traffic, incline.

In [ ]:
# TODO(owner: Q3) label the zero-emission lines (evidence list), join features from Q1 (lines_tbl), GTFS distance and depot; compare distributions

In [ ]:
# TODO(owner: Q3) external signals: traffic (e.g. Madrid traffic intensity open data) and incline (elevation along shapes); document sources with fetch()

## Q4 · How would we transition the buses? [ONCE]
**a) Grid electrification / depots:** depot information, gap to electrification per depot, route impact / coverage.
**b) Which routes are replaced:** phases, route impact (buses, bus metrics).

In [ ]:
# TODO(owner: Q4a) per depot: required MW (main notebook §6) vs connection MW -> gap; lines and buses covered

In [ ]:
# TODO(owner: Q4b) phases of lines by year for each timeline anchor; buses and km converted per phase

## Q5 · What does implementing electrification mean? [ONCE]
Finance / costs: bus purchases (`P['bus_price_eur']`), age and retirement of the gas fleet, gas cost, labour.

In [ ]:
# TODO(owner: Q5) retirements by year from the gas fleet's age profile (Q1); capex per timeline anchor; gas and labour cost sources

## Q6 · Scenario comparison [ONCE]
Compare the timeline anchors (2030 ideal · 2033 wildcard · 2035 mid · must-do year) on: depot → line, area / zones, bus ages / lifetime, distance, emissions, ridership.

In [ ]:
# TODO(owner: Q6) one row per anchor: buses bought per year, depots needing upgrades, zones covered, avg retirement age, km and riders on zero-emission lines, CO2 avoided

## Visualisation
Board list: scenario planning (2030 / 2035 / wildcard), map and routes with lines, bus scheduling (peak / off-peak, days of the week, times), future state over time (costs, emissions, line transition). The bus-scheduling view is below; the rest are TODO.

In [ ]:
by_hour = (supply.groupby(["TipoDiaMo", "FechaSer", "CodFranja"]).NumCCReal.sum()
           .groupby(["TipoDiaMo", "CodFranja"]).mean().rename("buses_in_service").reset_index())
by_hour["day_type"] = by_hour.TipoDiaMo.map({"LA": "Working day", "SA": "Saturday", "FE": "Sunday / holiday"})
fig = px.line(by_hour, x="CodFranja", y="buses_in_service", color="day_type", markers=True,
              title="Average buses in service by hour, Jan-Aug 2026", labels={"CodFranja": "hour band (24+ = after midnight)"})
fig.show()
peak_hour = by_hour.loc[by_hour[by_hour.TipoDiaMo == "LA"].buses_in_service.idxmax()]
print(f"Working-day peak: {peak_hour.buses_in_service:.0f} buses in hour band {peak_hour.CodFranja}")

In [ ]:
# TODO(owner: visualisation) scenario planning chart, route map coloured by status, future-state costs / emissions / line transition

## Checks, assumptions and sources

In [ ]:
display(pd.DataFrame(CHECKS)); display(pd.DataFrame(ASSUMPTIONS)); show_register()